# IMUSA: Multimodal Punjabi Meme Sentiment Analysis — Training & Inference Pipeline

This notebook trains a Late-Fusion Multimodal Model (**Vision Transformer + XLM-RoBERTa + Gated Fusion + Focal Loss**) for 4-class sentiment classification on Punjabi memes (`Sarcasm`, `Motivational`, `Neutral`, `Offensive`).

**Hardware**: Google Colab T4 GPU (Free tier)

## 1. Environment Setup & Repository Clone

In [ ]:
# 1. Environment & GPU Setup
!nvidia-smi
!pip install -q uv
import os
import sys

if not os.path.exists("imusa-multimodal-sentiment"):
    !git clone https://github.com/shubhojit-mitra-dev/imusa-multimodal-sentiment.git project
%cd /content/project
!git pull origin main
!pip install -e libs/imusa
sys.path.insert(0, "/content/project/libs/imusa/src")

## 2. Google Drive Integration & Automatic data.zip Handling

In [ ]:
import os
import shutil

from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)
gdrive_zip = "/content/drive/MyDrive/data.zip"
local_zip = "/content/project/data.zip"

if os.path.exists(gdrive_zip):
    print("Found data.zip in Google Drive. Copying locally...")
    shutil.copy(gdrive_zip, local_zip)
elif not os.path.exists(local_zip):
    print("data.zip not found in Google Drive (MyDrive/data.zip).")
    print("Please select and upload data.zip from your computer now:")
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(".zip"):
            shutil.move(fname, local_zip)
            break

# Copy to Google Drive for future runs
if os.path.exists(local_zip) and not os.path.exists(gdrive_zip):
    print("Saving data.zip to Google Drive (MyDrive/data.zip) for future runs...")
    try:
        shutil.copy(local_zip, gdrive_zip)
        print("Saved to Google Drive.")
    except Exception as e:
        print(f"Note: Could not copy to Drive: {e}")

# Unzip dataset
!unzip -q -o /content/project/data.zip -d /content/project/
print("Dataset extracted to data/.")

## 3. Execute Dataset Cleaning & EDA Pipeline

In [ ]:
!python scripts/clean_data.py
!python scripts/explore_data.py

## 4. Train Multimodal Model with Cosine Warmup & Focal Loss

In [ ]:
!python scripts/train.py --epochs 10 --batch-size 16 --lr 2e-5 --loss focal --warmup-ratio 0.1

## 5. Run Test Set Inference & Generate Submission CSV

In [ ]:
!python scripts/predict.py --checkpoint outputs/checkpoints/best_model.pt --output outputs/submission.csv

## 6. Download Results & Submission Artifacts

In [ ]:
from google.colab import files

files.download("outputs/submission.csv")
files.download("outputs/checkpoints/best_model.pt")